# A computational solution to the problem of evaluating poker-variant hands ranking

## The definition of the problem

In poker, there is the ranking of hands, according to the probability of a hand appearing in a round.

https://en.wikipedia.org/wiki/Poker_probability

One complication is that since in some popular variations of poker such as Texas hold 'em, a player uses the best five-card poker hand out of seven cards, the ranking slighly changes.

In Greece, the similar game is called poka and is most probably related to the French version. It is usually played with 32 cards (7 to King, plus Ace), and has many variations with respect the common cards on the table and they ways they are being layed out, the cards the players get, etc. Of course this is reflected on the rankings as well.

The general ranking in poka is the following:

- straight flush
- four of a kind
- flush
- full house
- straight
- three of a kind
- two pairs
- one pair

The rule sais that when the player can choose from 8 (inclusive) or more cards, or when there is a joker card, then `three of a kind` beats `straight` and `full house` beats `flush`. So the ranking becomes:

- straight flush
- four of a kind
- full house
- flush
- three of a kind
- straight
- two pairs
- one pair

The purpose here is to numerically simulate two games, one of each kind, and verify that these indeed follow the supposed rankings.


## Proposed solution

We need two different poka games in order to compute and compare the rankings. What we will do, is use the same game (one with 6 available cards to the players) and compute the ranking twice; once without and another time with a joker card.

The base game is simple:

- There are 3 cards on the table, available to all players (community cards)
- Each player has another 3 cards
- Out of these 6 cards, the player has to chose the `best` hand

We strip away all the rest of the game mechanics, bets, bluffs, as these do not influnce the base probability of a hand appearing.

Variation 1 - many cards:

- There are 5 cards on the table, available to all players (community cards)
- Each player has another 3 cards
- Out of these 8 cards, the player has to chose the `best` hand

Variation 2 - the joker:

- 7, the lowest ranking card, is a joker and can substitute for any other card.
- The rest of the game is the same as the base

Effectivelly, we have set up an experiment for comparing the base game against the two conditions (joker or many cards), that modify the ranking of the hands.


## Note on the joker card

The presense of a joker card modifies the hands themselves and not just their ranking.

With a joker a new hand is possible: five of a kind. This hand is supposed to be the strongest of all, so there is one more hypothesis to test.

### The deck of cards

In [1]:
"""Module docstring to keep pylint happy"""

## Imports
import itertools
from collections import Counter
from typing import Literal, NamedTuple
import ipytest
import pytest

## Configuration
ipytest.autoconfig()
RUN_TESTS = True

## Constants
DECK_SIZE = 32
POKA_HAND_LENGTH = 5
NUM_OF_JOKERS = 4
RANK_REPLACED_BY_JOKER = 7
JOKER_TUPLE = (0, "J")

In [2]:
## For playing poka, we need a deck of cards
## The numbers 11, 12, 13 represent the jack, queen, and king, respectivelly
## ["Hearts", "Diamonds", "Clubs", "Spades"] -> ["H", "D", "C", "S"]


class Card(NamedTuple):
    """A namedtuped that represent a playing card"""

    rank: int
    suit: str


cards_as_tuples = list(
    itertools.product([1, 7, 8, 9, 10, 11, 12, 13], ["H", "D", "C", "S"])
)

normal_deck = [Card(*a_card) for a_card in cards_as_tuples]

## don't need anymore
del cards_as_tuples

## replace each 7 with a joker card
joker_deck = [a_card for a_card in normal_deck if a_card.rank != RANK_REPLACED_BY_JOKER]
jokers = [Card(*JOKER_TUPLE)] * NUM_OF_JOKERS
joker_deck = joker_deck + jokers

In [3]:
%%ipytest

if RUN_TESTS:
    
    def test_normal_deck_size():
        assert len(normal_deck) == DECK_SIZE, f"The deck should have exactly {DECK_SIZE} cards"

    def test_joker_deck_size():
        assert len(joker_deck) == DECK_SIZE, f"The deck should have exactly {DECK_SIZE} cards"

    def test_joker_cards_number():
        num_of_jokers = len([a_card for a_card in joker_deck if a_card.suit == "J"])
        assert num_of_jokers == NUM_OF_JOKERS, f"The deck should have exactly {NUM_OF_JOKERS} jokers"
        

...                                                                                          [100%]
3 passed in 0.01s


### What hand do I have?

In [4]:
def n_of_a_kind(the_hand: list[Card], n_of: Literal[5, 4, 3, 2]) -> bool:
    """
    Args:
        the_hand: a list of 5 tuples; each tuple has two elements that represent a card;
            first comes the rank, then the suit. e.g. (9, 'H')
        n_of: the number of matching ranks.
            Allowed values 5,4,3,2 from five_of_a_kind to a single pair.
    Returns:
        a binary response, True/False.
            True if there are exactly `n_of` cards of the same rank, False otherwise.
    Raises:
        ValueError: If hand does not have exactly five cards
        ValueError: If `n_of` is not in the allowed set.
    Notes:
        five of a kind is only possible with a joker card deck - no joker, no 5 same ranks
    """

    if len(the_hand) != POKA_HAND_LENGTH:
        raise ValueError(f"The hand must have exactly {POKA_HAND_LENGTH} cards")

    allowed = {5, 4, 3, 2}
    if n_of not in allowed:
        raise ValueError(f"Invalid number of kind: '{n_of}'. Allowed: {allowed}")

    is_n_kind = False

    card_ranks = []
    for a_card in the_hand:
        card_ranks.append(a_card.rank)

    ## hand: [(1,D),(1,H),(9,H),(10,S),(10,D)] -> Counter({1: 2, 10: 2, 9: 1})
    rank_count = Counter(card_ranks)

    num_jokers = rank_count[Card(*JOKER_TUPLE).rank]

    ## remove jokers, if they exist, so that we now count normal cards
    try:
        rank_count.pop(Card(*JOKER_TUPLE).rank)
    except KeyError:
        pass

    ## most_common(1) returns a list of (one) tupple -> [(1, 3)]
    ## where 1 is the key (rank in our case) and 3 is the count
    ## we have removed the jokers so we do not double count them, if most common
    num_most_common_count = rank_count.most_common(1)[0][1]

    if num_jokers + num_most_common_count == n_of:
        is_n_kind = True

    return is_n_kind

In [5]:
%%ipytest

if RUN_TESTS:

    ###################### Assertions tests start ######################
    def test_short_hand():
        with pytest.raises(ValueError):
            n_of_a_kind( [Card(1,'H'), 
                       Card(1,'S')],
                       5)  

    def test_long_hand():
        with pytest.raises(ValueError):
            n_of_a_kind( [Card(1,'H'), 
                       Card(1,'S'), 
                       Card(1,'D'), 
                       Card(1,'C'), 
                       Card(10,'S'), 
                       Card(10,'S')],
                       5)  
            
    def test_n_of_restriction():
        with pytest.raises(ValueError):
            n_of_a_kind( [Card(1,'H'), 
                       Card(1,'S'), 
                       Card(1,'D'), 
                       Card(1,'C'), 
                       Card(10,'S')],
                       555555555555)
    ###################### Assertions tests end ######################

    ###################### 5 of a kind tests start ######################
    
    def test_four_plus_one():
        assert n_of_a_kind( [Card(1,'H'), 
                          Card(1,'S'), 
                          Card(1,'D'), 
                          Card(1,'C'), 
                          Card(0,'J')],
                            5
                       ) == True, f"This hand should be five of a kind"

    def test_three_plus_two():
        assert n_of_a_kind( [Card(1,'H'), 
                          Card(1,'S'), 
                          Card(1,'D'), 
                          Card(0,'J'), 
                          Card(0,'J')] ,
                            5
                       ) == True, f"This hand should be five of a kind"

    def test_no_five_kind():
        assert n_of_a_kind( [Card(12,'H'), 
                          Card(1,'S'), 
                          Card(1,'C'), 
                          Card(0,'J'), 
                          Card(0,'J')],
                            5
                       )  == False, f"This hand should NOT be five of a kind"
    ###################### 5 of a kind tests end ######################
        
    ###################### 4 of a kind tests start ######################
    def test_three_plus_one():
        assert n_of_a_kind( [Card(9,'H'), 
                          Card(1,'S'), 
                          Card(1,'D'), 
                          Card(1,'C'), 
                          Card(0,'J')],
                            4
                       ) == True, f"This hand should be four of a kind"

    def test_two_plus_two():
        assert n_of_a_kind( [Card(9,'H'), 
                          Card(1,'S'), 
                          Card(1,'D'), 
                          Card(0,'J'), 
                          Card(0,'J')],
                          4
                       ) == True, f"This hand should be four of a kind"

    def test_no_joker():
        assert n_of_a_kind( [Card(1,'H'), 
                          Card(1,'S'), 
                          Card(1,'C'), 
                          Card(1,'D'), 
                          Card(8,'C')],
                            4
                       )  == True, f"This hand should be four of a kind"

    def test_most_jokers():
        assert n_of_a_kind( [Card(9,'H'), 
                          Card(1,'S'), 
                          Card(0,'J'),
                          Card(0,'J'), 
                          Card(0,'J')],
                            4
                       )  == True, f"This hand should be four of a kind"

    def test_no_four_kind():
        assert n_of_a_kind( [Card(9,'H'), 
                          Card(1,'S'), 
                          Card(9,'C'), 
                          Card(1,'D'), 
                          Card(0,'J')],
                            4
                       )  == False, f"This hand should NOT be four of a kind"
    ###################### 4 of a kind tests end ######################
        
    ###################### 3 of a kind tests start ######################
    def test_exactly_three_kind():
        assert n_of_a_kind( [Card(1,'H'), 
                          Card(1,'S'), 
                          Card(1,'D'), 
                          Card(9,'C'), 
                          Card(10,'S')],
                            3
                       ) == True, f"This hand should be three of a kind"


    def test_exactly_three_kind_with_joker():
        assert n_of_a_kind( [Card(10,'H'),
                          Card(10,'S'),
                          Card(0,'J'), 
                          Card(8,'D'), 
                          Card(9,'C'), 
                          ],
                            3
                       ) == True, f"This hand should be three of a kind"
    
    def test_if_more_than_pair():
        assert n_of_a_kind( [Card(1,'H'), 
                          Card(1,'S'), 
                          Card(1,'D'), 
                          Card(0,'J'), 
                          Card(10,'S') ],
                            3
                       )  == False, f"This hand should NOT be three of a kind"
    ###################### 3 of a kind tests end ###################### 
    
    ###################### 2 of a kind tests start ######################
    def test_if_no_pair():
        assert n_of_a_kind( [Card(12,'H'), 
                          Card(1,'S'), 
                          Card(9,'C'), 
                          Card(10,'S'), 
                          Card(8,'C')],
                            2
                       )  == False, f"This hand should NOT be four of a kind"


    def test_exactly_one_pair():
        assert n_of_a_kind( [Card(1,'H'), 
                          Card(1,'S'), 
                          Card(8,'D'), 
                          Card(9,'C'), 
                          Card(10,'S')],
                            2
                       ) == True, f"This hand should contain exactly one pair"
    
    def test_exactly_one_pair_with_joker():
        assert n_of_a_kind( [Card(1,'H'), 
                          Card(0,'J'), 
                          Card(8,'D'), 
                          Card(9,'C'), 
                          Card(10,'S')],
                            2
                       ) == True, f"This hand should contain exactly one pair"
    
    def test_if_more_than_pair():
        assert n_of_a_kind( [Card(1,'H'), 
                          Card(1,'S'), 
                          Card(1,'D'), 
                          Card(9,'C'), 
                          Card(10,'S') ],
                            2
                       )  == False, f"This hand should NOT be one pair"

    def test_if_no_pair():
        assert n_of_a_kind( [Card(12,'H'), 
                          Card(1,'S'), 
                          Card(9,'C'), 
                          Card(10,'S'), 
                          Card(8,'C')],
                            2
                       )  == False, f"This hand does NOT contain a pair"
    ###################### 2 of a kind tests end ######################


.................                                                                            [100%]
17 passed in 0.03s


In [6]:
def flush(the_hand: list[Card]) -> bool:
    """
    Args:
        the_hand: a list of 5 tuples; each tuple has two elements that represent a card;
            first comes the rank, then the suit. e.g. (9, 'H')
    Returns:
        a binary response, True/False.
            True if there exactly 5 cards of the same suit, False otherwise.
    Raises:
        ValueError: If hand does not have exactly five cards
    """

    if len(the_hand) != POKA_HAND_LENGTH:
        raise ValueError(f"The hand must have exactly {POKA_HAND_LENGTH} cards")

    is_flush = False

    card_suits = []
    for a_card in the_hand:
        card_suits.append(a_card.suit)

    ## hand: [(1,D),(1,H),(9,H),(10,S),(10,D)] -> Counter({D: 2, H: 2, S: 1})
    suit_count = Counter(card_suits)

    num_jokers = suit_count[Card(*JOKER_TUPLE).suit]

    ## remove jokers, if they exist, so that we now count normal cards
    try:
        suit_count.pop(Card(*JOKER_TUPLE).suit)
    except KeyError:
        pass

    ## most_common(1) returns a list of (one) tupple -> [('C', 3)]
    ## where 'C' is the key (suit in our case) and 3 is the count
    ## we have removed the jokers so we do not double count them, if most common
    suit_most_common_count = suit_count.most_common(1)[0][1]

    if num_jokers + suit_most_common_count == 5:
        is_flush = True

    return is_flush

In [7]:
%%ipytest

if RUN_TESTS:


    ###################### Assertions tests start ######################
    def test_short_hand():
        with pytest.raises(ValueError):
            flush( [Card(1,'H'), 
                       Card(1,'S')]
                       )  

    def test_long_hand():
        with pytest.raises(ValueError):
            flush( [Card(1,'H'), 
                       Card(1,'S'), 
                       Card(1,'D'), 
                       Card(1,'C'), 
                       Card(10,'S'), 
                       Card(10,'S')]
                       )  
            
    ###################### Assertions tests end ######################

    ###################### is flush tests start ######################

    def test_flush_no_joker():
        assert flush( [Card(12,'H'), 
                          Card(1,'H'), 
                          Card(9,'H'), 
                          Card(10,'H'), 
                          Card(8,'H')]
                       )  == True, f"This hand is a flush"

    def test_flush_with_jokers():
        assert flush( [Card(12,'H'), 
                          Card(1,'H'), 
                          Card(9,'H'), 
                          Card(0,'J'), 
                          Card(0,'J')]
                       )  == True, f"This hand is a flush"
        
    def test_no_flush():
        assert flush( [Card(12,'H'), 
                          Card(1,'H'), 
                          Card(9,'C'), 
                          Card(0,'J'), 
                          Card(0,'J')]
                       )  == False, f"This hand is NOT a flush"

    ###################### is flush tests start ######################

.....                                                                                        [100%]
5 passed in 0.01s


In [8]:
def straight(the_hand: list[Card]) -> bool:
    """
    Args:
        the_hand: a list of 5 tuples; each tuple has two elements that represent a card;
            first comes the rank, then the suit. e.g. (9, 'H')
    Returns:
        a binary response, True/False.
            True if all cards are in numeric order, False otherwise.
    Raises:
        ValueError: If hand does not have exactly five cards
    """

    if len(the_hand) != POKA_HAND_LENGTH:
        raise ValueError(f"The hand must have exactly {POKA_HAND_LENGTH} cards")

    is_straight = False

    ## The Ace can be either lowest or highest card
    straight_sequences = []
    straight_sequences.append([1, 7, 8, 9, 10])
    straight_sequences.append([7, 8, 9, 10, 11])
    straight_sequences.append([8, 9, 10, 11, 12])
    straight_sequences.append([9, 10, 11, 12, 13])
    straight_sequences.append([10, 11, 12, 13, 1])

    card_ranks = []
    for a_card in the_hand:
        card_ranks.append(a_card.rank)

    ## We start with a possible sequence and try to rebuild it with the available cards
    for straight_sequence in straight_sequences:
        target = []
        available_cards = card_ranks.copy()

        for rank in straight_sequence:
            if rank in available_cards:
                target.append(rank)
                available_cards.remove(rank)
            elif (
                rank not in available_cards
                and Card(*JOKER_TUPLE).rank in available_cards
            ):
                target.append(rank)
                available_cards.remove(Card(*JOKER_TUPLE).rank)
            else:
                ## no need to continue if we cannot rebuilt (no cards or no joker)
                break

        if target in straight_sequences:
            is_straight = True
            ## if we found at least one straight sequence, we are done searching
            break

    return is_straight

In [9]:
%%ipytest

if RUN_TESTS:


    ###################### Assertions tests start ######################
    def test_short_hand():
        with pytest.raises(ValueError):
            straight( [Card(1,'H'), 
                       Card(1,'S')]
                       )  

    def test_long_hand():
        with pytest.raises(ValueError):
            straight( [Card(1,'H'), 
                       Card(1,'S'), 
                       Card(1,'D'), 
                       Card(1,'C'), 
                       Card(10,'S'), 
                       Card(10,'S')]
                       )  
            
    ###################### Assertions tests end ######################

    ###################### is flush tests start ######################

    def test_straight_no_joker():
        assert straight( [Card(7,'H'), 
                          Card(8,'D'), 
                          Card(9,'H'), 
                          Card(10,'H'), 
                          Card(1,'H')]
                       )  == True, f"This hand is a straight"

    def test_straight_with_joker():
        assert straight( [Card(7,'H'), 
                          Card(8,'D'), 
                          Card(9,'H'), 
                          Card(10,'H'), 
                          Card(0,'J')]
                       )  == True, f"This hand is a straight"

    def test_straight_with_two_jokers():
        assert straight( [Card(7,'H'), 
                          Card(8,'D'), 
                          Card(0,'J'), 
                          Card(10,'H'), 
                          Card(0,'J')]
                       )  == True, f"This hand is a straight"
    
        
    def test_no_straight():
        assert straight( [Card(12,'H'), 
                          Card(1,'H'), 
                          Card(9,'C'), 
                          Card(0,'J'), 
                          Card(0,'J')]
                       )  == False, f"This hand is NOT a straight"

    ###################### is flush tests start ######################

......                                                                                       [100%]
6 passed in 0.01s


In [10]:
def straight_flush(the_hand: list[Card]) -> bool:
    """
    Args:
        the_hand: a list of 5 tuples; each tuple has two elements that represent a card;
            first comes the rank, then the suit. e.g. (9, 'H')
    Returns:
        a binary response, True/False.
            True if all cards are in numeric order and of the same suit, False otherwise.
    Raises:
        ValueError: If hand does not have exactly five cards
    """

    if len(the_hand) != POKA_HAND_LENGTH:
        raise ValueError(f"The hand must have exactly {POKA_HAND_LENGTH} cards")

    is_straight_flush = False

    if straight(the_hand) == flush(the_hand) == True:
        is_straight_flush = True

    return is_straight_flush

In [11]:
%%ipytest

if RUN_TESTS:


    ###################### Assertions tests start ######################
    def test_short_hand():
        with pytest.raises(ValueError):
            straight_flush( [Card(1,'H'), 
                       Card(1,'S')]
                       )  

    def test_long_hand():
        with pytest.raises(ValueError):
            straight_flush( [Card(1,'H'), 
                       Card(1,'S'), 
                       Card(1,'D'), 
                       Card(1,'C'), 
                       Card(10,'S'), 
                       Card(10,'S')]
                       )  
            
    ###################### Assertions tests end ######################

    ###################### is  straight flush tests start ######################

    def test_straight_flush_no_joker():
        assert straight_flush( [Card(7,'H'), 
                                  Card(8,'H'), 
                                  Card(9,'H'), 
                                  Card(10,'H'), 
                                  Card(1,'H')]
                               )  == True, f"This hand is a straight flush"

    def test_straight_flush_with_joker():
        assert straight_flush( [Card(7,'H'), 
                                  Card(8,'H'), 
                                  Card(9,'H'), 
                                  Card(10,'H'), 
                                  Card(0,'J')]
                               )  == True, f"This hand is a straight flush"
        
    def test_no_straight_flush():
        assert straight_flush( [Card(12,'H'), 
                              Card(1,'H'), 
                              Card(9,'C'), 
                              Card(0,'J'), 
                              Card(0,'J')]
                           )  == False, f"This hand is NOT a straight flush"
        
    ###################### is  straight flush tests end ######################

.....                                                                                        [100%]
5 passed in 0.02s
